# Aula 1 - Mastering Machine Learning Advanced

## Feature Engineering & Pré-processamento de Dados

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. IBM Telco Customer Churn — https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113

## Índice

1. [Introdução](#1)
2. [Carregamento e Inspeção dos Dados](#2)
3. [Tratamento de Valores Ausentes](#3)
   - 3.1 [Imputação por Mediana e Moda](#31)
   - 3.2 [Imputação por Grupo](#32)
   - 3.3 [KNN Imputer](#33)
4. [Detecção e Tratamento de Outliers](#4)
   - 4.1 [Critério IQR](#41)
   - 4.2 [Z-Score](#42)
5. [Encoding de Variáveis Categóricas](#5)
   - 5.1 [Label Encoding](#51)
   - 5.2 [One-Hot Encoding](#52)
   - 5.3 [Target Encoding](#53)
6. [Normalização e Padronização](#6)
7. [Criação de Novas Features](#7)
8. [Feature Selection](#8)
   - 8.1 [Correlação de Pearson](#81)
   - 8.2 [SelectKBest](#82)
   - 8.3 [Feature Importance com Random Forest](#83)
9. [Pipeline Completo](#9)
10. [Conclusão](#10)

# 1. Introdução <a id="1"></a>

**Feature Engineering** é o processo de usar o conhecimento do domínio para criar, transformar e selecionar as variáveis que alimentam nossos modelos de Machine Learning.

Na prática, a qualidade dos dados de entrada tem muito mais impacto no resultado final do que a escolha do algoritmo. Um modelo simples com boas features supera quase sempre um modelo complexo com dados mal preparados.

![Feature Engineering Pipeline](https://i.imgur.com/JmXLDfZ.png)

**Problema de negócio:** vamos trabalhar com o dataset de **Churn de clientes de uma operadora de telecomunicações**. O objetivo é prever quais clientes têm maior probabilidade de cancelar o serviço — um problema clássico e de altíssimo valor para o negócio.

Ao longo desta aula vamos cobrir:

- **Tratamento de dados faltantes** — o que fazer quando o dataset tem buracos
- **Outliers** — identificar e tratar valores que fogem do padrão
- **Encoding** — transformar categorias em números de forma inteligente
- **Scaling** — colocar todas as features na mesma escala
- **Criação de features** — combinar variáveis existentes para criar informação nova
- **Feature Selection** — escolher apenas as features que realmente importam

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)

# 2. Carregamento e Inspeção dos Dados <a id="2"></a>

O dataset contém informações sobre ~7.000 clientes de uma operadora, incluindo dados de serviços contratados, dados financeiros e se o cliente cancelou ou não (coluna `Churn`).

In [ ]:
# carregando o dataset diretamente do GitHub
url = 'https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv'
df = pd.read_csv(url)
df.head()

In [ ]:
print(f'Shape: {df.shape}')
df.info()

Observação importante: `TotalCharges` está como `object`, mas deveria ser numérico. Isso acontece porque existem registros com espaço em branco no lugar do valor. Vamos corrigir.

In [ ]:
# convertendo TotalCharges para numérico — valores inválidos viram NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# verificando valores ausentes
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
df.describe()

In [ ]:
# taxa de churn no dataset
churn_rate = df['Churn'].value_counts(normalize=True) * 100
print(churn_rate)

churn_rate.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='white', figsize=(6, 4))
plt.title('Distribuição de Churn')
plt.xlabel('Churn')
plt.ylabel('%')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 3. Tratamento de Valores Ausentes <a id="3"></a>

Os 11 registros com `TotalCharges` ausente correspondem a clientes com `tenure = 0` (clientes que entraram mas ainda não tiveram fatura). Antes de decidir o que fazer, precisamos entender o padrão.

In [ ]:
# analisando os registros com TotalCharges ausente
df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head()

In [ ]:
# mapa de calor dos dados ausentes
plt.figure(figsize=(12, 3))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Mapa de Valores Ausentes')
plt.show()

## 3.1 Imputação por Mediana e Moda <a id="31"></a>

Para a maioria dos casos com poucos ausentes, imputar pela mediana (numérico) ou moda (categórico) é a estratégia mais rápida e eficaz.

In [ ]:
df_clean = df.copy()

# imputando TotalCharges pela mediana
mediana_total = df_clean['TotalCharges'].median()
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(mediana_total)

print(f'Mediana usada para imputação: R$ {mediana_total:.2f}')
print(f'Valores ausentes restantes: {df_clean["TotalCharges"].isnull().sum()}')

## 3.2 Imputação por Grupo <a id="32"></a>

Uma estratégia mais inteligente é imputar pela mediana **dentro de cada grupo**. No contexto de telecom, faz mais sentido comparar clientes com o mesmo tipo de contrato.

In [ ]:
# imputando pela mediana de TotalCharges dentro de cada tipo de contrato
df_grupo = df.copy()
df_grupo['TotalCharges'] = df_grupo.groupby('Contract')['TotalCharges'].transform(
    lambda x: x.fillna(x.median())
)

print('Mediana de TotalCharges por tipo de contrato:')
print(df_grupo.groupby('Contract')['TotalCharges'].median())

## 3.3 KNN Imputer <a id="33"></a>

Para dados mais complexos, o **KNN Imputer** preenche o valor ausente com base nos K vizinhos mais próximos — leva em conta o contexto de cada registro, não apenas a estatística global.

In [ ]:
# demonstração do KNN Imputer com as colunas numéricas
cols_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_knn = df[cols_numericas].copy()

imputer_knn = KNNImputer(n_neighbors=5)
df_knn_imputed = pd.DataFrame(
    imputer_knn.fit_transform(df_knn),
    columns=cols_numericas
)

print(f'Valores ausentes antes: {df_knn.isnull().sum().sum()}')
print(f'Valores ausentes depois: {df_knn_imputed.isnull().sum().sum()}')

**Quando usar cada estratégia:**

| Estratégia | Quando usar |
|---|---|
| Mediana global | Poucos ausentes, dados com outliers |
| Moda | Variáveis categóricas |
| Imputação por grupo | Quando existe uma variável de segmentação relevante |
| KNN Imputer | Quando o padrão de ausência é mais complexo e há correlação entre features |

# 4. Detecção e Tratamento de Outliers <a id="4"></a>

Clientes com faturas mensais muito altas ou contratos atípicos podem distorcer o modelo. Precisamos identificá-los antes de treinar.

## 4.1 Critério IQR <a id="41"></a>

In [ ]:
# visualizando a distribuição das variáveis numéricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(y=df_clean[col], ax=ax, color='steelblue')
    ax.set_title(col)

plt.suptitle('Distribuição e Outliers — Dataset Telecom', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def detectar_outliers_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    return outliers, lim_inf, lim_sup

for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(df_clean[col])
    print(f'{col}: {len(outliers)} outliers | limites [{lim_inf:.2f}, {lim_sup:.2f}]')

## 4.2 Z-Score <a id="42"></a>

O Z-Score mede quantos desvios padrão um valor está da média. Valores com |Z| > 3 são candidatos fortes a outlier.

In [ ]:
# calculando z-score para MonthlyCharges
z_scores = np.abs(stats.zscore(df_clean['MonthlyCharges'].dropna()))
outliers_z = np.sum(z_scores > 3)

print(f'Outliers detectados pelo Z-Score (|Z| > 3): {outliers_z}')

# comparando IQR vs Z-Score visualmente
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# IQR
_, lim_inf, lim_sup = detectar_outliers_iqr(df_clean['MonthlyCharges'])
axes[0].hist(df_clean['MonthlyCharges'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(lim_inf, color='red', linestyle='--', label=f'Limite IQR: {lim_inf:.0f}')
axes[0].axvline(lim_sup, color='red', linestyle='--', label=f'Limite IQR: {lim_sup:.0f}')
axes[0].set_title('MonthlyCharges — Limites IQR')
axes[0].legend()

# Z-Score
media = df_clean['MonthlyCharges'].mean()
std = df_clean['MonthlyCharges'].std()
axes[1].hist(df_clean['MonthlyCharges'], bins=40, color='coral', edgecolor='white')
axes[1].axvline(media - 3*std, color='darkred', linestyle='--', label=f'Z=-3: {media-3*std:.0f}')
axes[1].axvline(media + 3*std, color='darkred', linestyle='--', label=f'Z=+3: {media+3*std:.0f}')
axes[1].set_title('MonthlyCharges — Limites Z-Score')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# estratégia de capping: limitar ao percentil 99
p99 = df_clean['TotalCharges'].quantile(0.99)
df_clean['TotalCharges_capped'] = df_clean['TotalCharges'].clip(upper=p99)

print(f'Máximo original:  {df_clean["TotalCharges"].max():.2f}')
print(f'Máximo pós-capping: {df_clean["TotalCharges_capped"].max():.2f}')

# 5. Encoding de Variáveis Categóricas <a id="5"></a>

O dataset de Telecom tem muitas variáveis categóricas — `gender`, `Contract`, `PaymentMethod`, `InternetService`, entre outras. Precisamos convertê-las para que os algoritmos consigam processar.

## 5.1 Label Encoding <a id="51"></a>

Adequado para variáveis binárias (`Yes/No`) ou ordinais.

In [ ]:
df_enc = df_clean.copy()

# colunas binárias: Yes/No → 1/0
colunas_binarias = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

for col in colunas_binarias:
    df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0})

# gender: Female/Male → 0/1
df_enc['gender'] = df_enc['gender'].map({'Female': 0, 'Male': 1})

df_enc[['gender', 'Partner', 'Dependents', 'PhoneService', 'Churn']].head()

## 5.2 One-Hot Encoding <a id="52"></a>

Para variáveis nominais com mais de 2 categorias — `Contract`, `PaymentMethod`, `InternetService` — usamos One-Hot Encoding.

In [ ]:
# one-hot encoding nas colunas com múltiplas categorias
colunas_ohe = ['Contract', 'PaymentMethod', 'InternetService']

df_enc = pd.get_dummies(df_enc, columns=colunas_ohe, drop_first=True)

print(f'Shape antes do OHE: {df_clean.shape}')
print(f'Shape depois do OHE: {df_enc.shape}')

## 5.3 Target Encoding <a id="53"></a>

Para `MultipleLines`, `OnlineSecurity`, `TechSupport` e variáveis similares — que têm 3 categorias incluindo `No service` — podemos usar Target Encoding: substituir cada categoria pela taxa de Churn daquele grupo.

In [ ]:
# target encoding para MultipleLines
target_enc_map = df.groupby('MultipleLines')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
)
print('Taxa de Churn por categoria de MultipleLines:')
print(target_enc_map)

df_enc['MultipleLines_te'] = df['MultipleLines'].map(target_enc_map)

# 6. Normalização e Padronização <a id="6"></a>

No contexto de Telecom, as features numéricas têm escalas muito diferentes: `tenure` varia de 0 a 72 meses, `MonthlyCharges` de R$18 a R$118, e `TotalCharges` de R$18 a R$8.684. Sem scaling, um modelo como KNN ou Regressão Logística com regularização vai dar peso desproporcional a `TotalCharges`.

In [ ]:
features_num = df_clean[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()

scaler_std = StandardScaler()
scaler_mm  = MinMaxScaler()
scaler_rob = RobustScaler()

feat_std = pd.DataFrame(scaler_std.fit_transform(features_num), columns=features_num.columns)
feat_mm  = pd.DataFrame(scaler_mm.fit_transform(features_num),  columns=features_num.columns)
feat_rob = pd.DataFrame(scaler_rob.fit_transform(features_num), columns=features_num.columns)

# comparação visual para MonthlyCharges
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

dados_plot = [
    (features_num['MonthlyCharges'], 'Original', 'steelblue'),
    (feat_std['MonthlyCharges'],     'StandardScaler', 'coral'),
    (feat_mm['MonthlyCharges'],      'MinMaxScaler', 'seagreen'),
    (feat_rob['MonthlyCharges'],     'RobustScaler', 'mediumpurple'),
]

for ax, (data, title, color) in zip(axes, dados_plot):
    ax.hist(data, bins=40, color=color, edgecolor='white')
    ax.set_title(title)

plt.suptitle('Comparação de Scalers — MonthlyCharges', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print('Estatísticas após StandardScaler:')
print(feat_std.describe().round(2))

**Regra prática para Telecom:**

| Situação | Scaler recomendado |
|---|---|
| Regressão Logística, SVM, Redes Neurais | StandardScaler |
| Algoritmos baseados em distância (KNN) | MinMaxScaler |
| Faturas com clientes corporativos (outliers altos) | RobustScaler |
| Árvores de Decisão, Random Forest, XGBoost | Nenhum — não precisam de scaling |

# 7. Criação de Novas Features <a id="7"></a>

A parte mais criativa do Feature Engineering. O dataset de Telecom oferece ótimas oportunidades para criar features com significado de negócio.

In [ ]:
df_fe = df_clean.copy()

# faixa de tempo como cliente (em anos)
df_fe['tenure_group'] = pd.cut(
    df_fe['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1 ano', '1-2 anos', '2-4 anos', '4+ anos']
)

# conta o número de serviços adicionais contratados
servicos = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
df_fe['num_servicos'] = df_fe[servicos].apply(
    lambda row: sum(v == 'Yes' for v in row), axis=1
)

# razão entre fatura total e mensal (aproxima quantos meses o cliente está ativo)
df_fe['charges_ratio'] = df_fe['TotalCharges'] / (df_fe['MonthlyCharges'] + 1)

# cliente sem nenhum serviço adicional
df_fe['sem_servicos_extra'] = (df_fe['num_servicos'] == 0).astype(int)

df_fe[['tenure', 'tenure_group', 'num_servicos', 'charges_ratio', 'sem_servicos_extra']].head(8)

In [ ]:
# analisando a taxa de churn por faixa de tempo como cliente
churn_by_tenure = df_fe.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

churn_by_tenure.plot(
    kind='bar', figsize=(8, 4), color='coral', edgecolor='white'
)
plt.title('Taxa de Churn por Tempo como Cliente')
plt.xlabel('Tempo como Cliente')
plt.ylabel('Taxa de Churn (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# taxa de churn por número de serviços adicionais
churn_by_servicos = df_fe.groupby('num_servicos')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

churn_by_servicos.plot(
    kind='bar', figsize=(10, 4), color='steelblue', edgecolor='white'
)
plt.title('Taxa de Churn por Número de Serviços Adicionais')
plt.xlabel('Número de Serviços')
plt.ylabel('Taxa de Churn (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Resultado claro: clientes com menos tempo de casa e menos serviços contratados têm maior probabilidade de churn. Esse é exatamente o tipo de insight que o Feature Engineering revela e que o modelo sozinho não capturaria com as features brutas.

# 8. Feature Selection <a id="8"></a>

Com ~20 colunas originais + OHE + features criadas, temos mais de 30 variáveis. Vamos identificar quais realmente contribuem para prever o Churn.

## 8.1 Correlação de Pearson <a id="81"></a>

In [ ]:
# preparando uma versão numérica do df para a análise de correlação
df_corr = df_fe[['tenure', 'MonthlyCharges', 'TotalCharges',
                 'num_servicos', 'charges_ratio', 'sem_servicos_extra', 'SeniorCitizen']].copy()
df_corr['Churn'] = (df_fe['Churn'] == 'Yes').astype(int)

corr_matrix = df_corr.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Matriz de Correlação de Pearson — Telecom Churn')
plt.tight_layout()
plt.show()

In [ ]:
# correlação de cada feature com o target Churn
cor_target = corr_matrix['Churn'].drop('Churn').abs().sort_values(ascending=False)

cor_target.plot(kind='barh', figsize=(8, 5), color='steelblue', edgecolor='white')
plt.title('Correlação com Churn (valor absoluto)')
plt.xlabel('Correlação')
plt.tight_layout()
plt.show()

## 8.2 SelectKBest <a id="82"></a>

In [ ]:
X_sel = df_corr.drop(columns=['Churn'])
y_sel = df_corr['Churn']

selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X_sel, y_sel)

scores = pd.Series(selector.scores_, index=X_sel.columns).sort_values(ascending=False)
scores.plot(kind='barh', figsize=(8, 5), color='coral', edgecolor='white')
plt.title('Score das Features — SelectKBest (F-score ANOVA)')
plt.xlabel('Score')
plt.tight_layout()
plt.show()

## 8.3 Feature Importance com Random Forest <a id="83"></a>

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_sel, y_sel)

importancias = pd.Series(rf.feature_importances_, index=X_sel.columns).sort_values(ascending=False)

importancias.plot(kind='barh', figsize=(8, 5), color='seagreen', edgecolor='white')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

# 9. Pipeline Completo <a id="9"></a>

Na prática, todas as etapas de pré-processamento precisam ser aplicadas de forma idêntica no treino e no teste. O `Pipeline` do scikit-learn garante isso — nenhuma transformação vaza do treino para o teste, evitando **data leakage**.

In [ ]:
# preparando o dataset para o pipeline
df_pipe = df.copy()
df_pipe['TotalCharges'] = pd.to_numeric(df_pipe['TotalCharges'], errors='coerce')
df_pipe['Churn'] = (df_pipe['Churn'] == 'Yes').astype(int)
df_pipe = df_pipe.drop(columns=['customerID'])

# features que vamos usar
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

X = df_pipe[cols_num + cols_cat]
y = df_pipe['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# pipeline para features numéricas
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# pipeline para features categóricas
pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# combinando os dois pipelines
preprocessor = ColumnTransformer([
    ('num', pipeline_num, cols_num),
    ('cat', pipeline_cat, cols_cat)
])

# pipeline final
pipeline_final = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

pipeline_final.fit(X_train, y_train)
y_pred = pipeline_final.predict(X_test)

print('=== Resultado com Pipeline Completo ===')
print(classification_report(y_test, y_pred, target_names=['Não Churn', 'Churn']))

In [ ]:
# comparativo: baseline (só features numéricas, sem tratamento) vs. pipeline completo
df_base = df_pipe[cols_num + ['Churn']].dropna()
X_b = df_base.drop(columns=['Churn'])
y_b = df_base['Churn']

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42)

lr_base = LogisticRegression(max_iter=1000)
lr_base.fit(X_tr_b, y_tr_b)

acc_base = accuracy_score(y_te_b, lr_base.predict(X_te_b))
acc_fe   = accuracy_score(y_test, y_pred)

print(f'Baseline (só features numéricas, sem FE):  {acc_base:.4f}')
print(f'Com Feature Engineering completo:          {acc_fe:.4f}')
print(f'Ganho:                                     +{(acc_fe - acc_base)*100:.2f}pp')

# 10. Conclusão <a id="10"></a>

Nesta aula cobrimos o ciclo completo de Feature Engineering aplicado a um problema real de churn em Telecom:

| Técnica | Aplicação no contexto Telecom |
|---|---|
| Conversão de tipos | `TotalCharges` como objeto → numérico |
| Imputação por mediana | `TotalCharges` com 11 registros ausentes |
| Imputação por grupo | Mediana por tipo de contrato |
| KNN Imputer | Imputação contextualizada nas features numéricas |
| Outliers IQR + Z-Score | Faturas atípicas de clientes corporativos |
| Label Encoding | Colunas binárias Yes/No |
| One-Hot Encoding | Contract, PaymentMethod, InternetService |
| Target Encoding | MultipleLines e serviços com 3 categorias |
| StandardScaler | Preparação para Regressão Logística |
| Feature criada: `tenure_group` | Segmentação por maturidade do cliente |
| Feature criada: `num_servicos` | Profundidade de relacionamento com a operadora |
| Feature criada: `charges_ratio` | Indicador de consistência de faturamento |
| Feature Selection | Correlação, F-score, Feature Importance |
| Pipeline | Garantia de reprodutibilidade treino/teste |

**Regra de ouro:** nunca aplique transformações do treino diretamente no teste. Use sempre um `Pipeline` para garantir consistência e evitar data leakage.

Na próxima aula vamos aprender a **avaliar modelos de verdade** com Cross-Validation e a encontrar os melhores hiperparâmetros com Grid Search e Optuna.

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)